# <font color="orange">Aiko Chat Tutorial: Create a Virtual Robot</font>

This tutorial shows how to create a simple virtual robot. The interface is identical to what we would use for a physical robot, so in theory you could replace this notebook with a real robot.

It is defined as an Actor in [Aiko Services](https://github.com/geekscape/aiko_services) which supports discoverability.

<font color="orange"><i>2026-06-08: version 0</i></font>

## <font color="orange">Running this against the aiko_turtle sandbox (2026-11)</font>

This is the June 2026 tutorial notebook, unchanged except for the broker name and these notes. It is a virtual turtle (drawn with ColabTurtlePlus inside the notebook) speaking the `turtle_robot:0` interface. The sandbox at https://turtle.aiko.hashtag.ai recognises that interface in the Registrar and shows your robot as a **twin** on its map, dead reckoning its position from the commands it receives, with a dashed ring, for as long as the notebook runs:

1. Below, set `_ACTOR_TYPE = "my_turtle_yourname"` to a name nobody else will pick: it is the Registrar name, and the twin's name.
2. Run every cell. Within a few seconds the twin appears on the sandbox map. Drag the twin to where you imagine the robot is.
3. Drive it from the part-1 notebook (with `_ROBOT_NAME` set to the same name), from the sandbox chat (`@@my_turtle_yourname forward 100`), or from the page's *aiko* layer. The notebook's turtle and the twin move together.

Run it in Google Colab: File > Upload notebook (or open it from GitHub: `anjsimmo/aiko_turtle`, path `examples/notebooks`). Runtime > Run all. If the broker name in your copy still says `mqtt.hashtag.ai`, change it to `aiko.hashtag.ai`.


## <font color="orange">Install the Aiko Services Python package</font>

In [ ]:
%cd /content
!git clone https://github.com/geekscape/aiko_services.git
%cd /content/aiko_services
!echo
!du -sh .
%ls

In [ ]:
# This cell takes around 20 seconds to run

!sed -i '/Pillow/d' pyproject.toml  # Avoid session restart due to older package version
!sed -i '/psutil/d' pyproject.toml  # Ditto
%pip install .

## <font color="orange">Create your own Virtual Robot that runs on a background thread</font>

Classes that extend Actor are discoverable by other Actors in Aiko. In this case, we'll put any messages we receive into a queue and decide what to do with them later.


In [ ]:
import socket
DEFAULT_USERNAME = f"{socket.gethostname()[:7]}" # default username, by default will be colab notebook id

In [ ]:
### CHANGE YOUR USER NAME HERE, DO NOT USE QUOTES ###
!echo Change USER or it will appear as $DEFAULT_USERNAME
%env USER=$DEFAULT_USERNAME

%env AIKO_LOG_LEVEL=DEBUG
%env AIKO_MQTT_HOST=aiko.hashtag.ai
%cd /content/aiko_services/src/aiko_services/examples/colab

In [ ]:
import aiko_services as aiko
from abc import abstractmethod

class TurtleRobot(aiko.Actor):
    @abstractmethod
    def action(self, value):
        pass

    @abstractmethod
    def forward(self, value):
        pass

    @abstractmethod
    def left(self, value):
        pass

    @abstractmethod
    def right(self, value):
        pass

    @abstractmethod
    def backward(self, value):
        pass


In [ ]:
# This cell creates the actor running on a background thread

import os
from queue import Queue
from threading import Thread

tags = ["ec=true", "type=robot", "commands=action,forward,left,right,backward"] # you can set any tags you like to make it easy to search for in the registrar

# Aiko services uses the composition pattern to create a service
_VERSION = 0
_ACTOR_TYPE = "turtle_robot" # IMPORTANT: Rename your virtual robot to something unique so you can find it
_PROTOCOL = f"{aiko.SERVICE_PROTOCOL_AIKO}/{_ACTOR_TYPE}:{_VERSION}"
init_args = aiko.actor_args(_ACTOR_TYPE, protocol=_PROTOCOL, tags=tags)

import queue
q = queue.Queue()


class TurtleRobotImp(aiko.Actor):
    def __init__(self, context: TurtleRobot):
        context.call_init(self, "Actor", context)

    def action(self, value):
        print(f"action {value}")
        q.put(("action", value))

    def forward(self, value):
        print(f"forward {value}")
        q.put(("forward", value))

    def left(self, value):
        print(f"left {value}")
        q.put(("left", value))

    def right(self, value):
        print(f"right {value}")
        q.put(("right", value))

    def backward(self, value):
        print(f"backward {value}")
        q.put(("backward", value))

robot = aiko.compose_instance(TurtleRobotImp, init_args)

thread = Thread(target=robot.run, args=(False,))
thread.daemon = True
thread.start()

In [ ]:
# The below information will be helpful for other services to discover the robot

import os
print("USER:", os.environ["USER"])
print("HOST:", socket.gethostname())
print("ACTOR_TYPE:", _ACTOR_TYPE)
print("PROTOCOL:", _PROTOCOL)
print("TAGS:", tags)


## <font color="orange">React to messages</font>

In [ ]:
!pip install ColabTurtlePlus

In [ ]:
from ColabTurtlePlus.Turtle import *
import json

clearscreen()
setup(400, 400)
shape("turtle")
speed(7)

while True:
    item = q.get()
    print(f"Received: {item}")
    (action, value) = item

    match action:
      case "action":
        match value:
          case "penup":
            penup()
          case "pendown":
            pendown()
          case "forward":
            forward(100)
          case "left":
            left(90)
          case "right":
            right(90)
          case "backward":
            backward(100)

      case "forward":
        forward(float(value))

      case "left":
        left(float(value))

      case "right":
        right(float(value))

      case "backward":
        backward(float(value))

    q.task_done()